<a href="https://colab.research.google.com/github/sandrokhizanishvili/AML_GNN_GMA/blob/main/baseline_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GINe + RWPE for Anti-Money Laundering Detection

**Course:** Graph Mining and Applications, Sapienza University of Rome

---

## What This Notebook Does

This notebook trains **GINe augmented with Random Walk Positional Encoding (RWPE)**.
It is the direct ablation counterpart to `gine_gfp_rwpe.ipynb`, isolating the effect
of RWPE alone on GINe (without GFP features).

RWPE(node_i, k) = probability that a random walk starting at node i returns in k steps.
walk_length=8 covers all major AML cycle patterns.

| Property | Value |
|----------|-------|
| Base architecture | GINe (GINEConv + edge readout) |
| Node feature dims | 5 (original) + 8 (RWPE) = 13 |
| Edge feature dims | 16 (baseline, no GFP) |


## 1. Installation

This cell installs the correct versions of PyTorch and PyTorch Geometric for Colab.


If the libraries are already installed correctly, you can skip this cell.

In [1]:
print("🚀 SWITCHING TO PYTORCH 2.8 (FAST MODE)...")

# 1. Uninstall the current mismatching versions
# We remove the one you just spent 15 mins compiling, because re-installing
# the CORRECT version via wheels will take only 30 seconds.
# os.system("pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib")
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib

# 2. Install PyTorch 2.8.0 (with CUDA 12.6 support)
# We specify the version explicitly to match the PyG documentation you found.
print("⬇️ Installing PyTorch 2.8.0...")
# os.system("pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126")
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

# 3. Install Graph Libraries for PyTorch 2.8
# This link matches the table in your screenshot: torch-2.8.0 + cu126
print("⬇️ Installing Graph Libraries (Wheels)...")
# os.system("pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.8.0+cu126.html")
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html

print("="*40)
print("✅ SETUP COMPLETE.")
print("⚠️ YOU MUST RESTART THE RUNTIME NOW (Runtime -> Restart Session)")
print("="*40)




import torch

try:
    import torch_sparse
    sparse_status = "✅ Installed"
    sparse_version = torch_sparse.__version__
except ImportError:
    sparse_status = "❌ Not Found"
    sparse_version = "N/A"

print(f"PyTorch Version:      {torch.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()}")
print(f"Torch Sparse Status:  {sparse_status} ({sparse_version})")

if torch.cuda.is_available() and sparse_status == "✅ Installed":
    print("\nSUCCESS! You are ready to run the training loop.")
else:
    print("\n⚠️ Something is still missing. Did you Restart the Runtime?")



!pip install torch_geometric

🚀 SWITCHING TO PYTORCH 2.8 (FAST MODE)...
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
⬇️ Installing PyTorch 2.8.0...
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 176.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 206.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 179.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 77.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 102.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/

## 2. Imports and Reproducibility

We import all required libraries and fix all random seeds so that results are reproducible across runs.

The main libraries used are:
- **PyTorch** for model definition and training
- **PyTorch Geometric (PyG)** for graph data structures and GNN layers
- **scikit-learn** for evaluation metrics
- **tqdm** for progress bars during training

In [2]:
# ── Standard library ──────────────────────────────────────────────────────────
import os, time, math, random, warnings
from google.colab import drive
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, precision_recall_curve, average_precision_score, matthews_corrcoef, auc
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

import torch_geometric
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import GINEConv
from torch_geometric.utils import degree
from torch_geometric.transforms import AddRandomWalkPE  # NEW: RWPE

from tqdm import tqdm

print(f'PyTorch          : {torch.__version__}')
print(f'PyTorch Geometric: {torch_geometric.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device           : {device}')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


PyTorch          : 2.8.0+cu126
PyTorch Geometric: 2.7.0
Device           : cuda


## 3. Loading the Graph Data

The graph was built in `Data_prepration.ipynb` and saved as three PyG `Data` objects. We load them here.

### Why Three Separate Graphs?

We use a **cumulative snapshot** design that follows the paper's protocol:

- `train_graph` contains only the training period edges. All of them are labelled and used for training.
- `val_graph` contains training + validation period edges. Only the new validation edges are evaluated; the training edges provide neighbourhood context for message passing.
- `test_graph` contains all edges from all periods. Only the test period edges are evaluated.

This design is important because financial transactions exist in a temporal context. A suspicious transaction in the test set should be evaluated using knowledge of the account's full history, not just the test period. By including all prior edges in each snapshot, we give the GNN access to that historical context during message passing.

### Data Split (Temporal)

The split is done chronologically to prevent data leakage:
- **Train:** first 60% of time period
- **Validation:** next 20%
- **Test:** final 20%

In [3]:
# # Mount Google Drive where the graph files and model checkpoints are stored

# drive.mount('/content/drive')

# os.chdir('/content/drive/MyDrive/GMA_GNN_AML')

In [4]:
DATA_DIR = '/kaggle/input/datasets/sandrokhizanishvili/aml-gnn-pna/Data'

train_graph = torch.load(os.path.join(DATA_DIR, 'train_graph.pt'), weights_only=False)
val_graph   = torch.load(os.path.join(DATA_DIR, 'val_graph.pt'),   weights_only=False)
test_graph  = torch.load(os.path.join(DATA_DIR, 'test_graph.pt'),  weights_only=False)

# ── RWPE injection ────────────────────────────────────────────────────────────
# Applied per-snapshot using only that snapshot's edges (no temporal leakage).
# walk_length=8 captures AML cycles up to length 8.
# Result: node feat dim 5 -> 13.
RWPE_WALK_LENGTH = 8

def add_rwpe(graph, walk_length=RWPE_WALK_LENGTH):
    transform = AddRandomWalkPE(walk_length=walk_length, attr_name='rwpe')
    graph = transform(graph)
    graph.x = torch.cat([graph.x, graph.rwpe], dim=-1)
    del graph.rwpe
    return graph

print('Computing RWPE per snapshot...')
train_graph = add_rwpe(train_graph); print(f'  train: node_dim={train_graph.x.shape[1]}')
val_graph   = add_rwpe(val_graph);   print(f'  val:   node_dim={val_graph.x.shape[1]}')
test_graph  = add_rwpe(test_graph);  print(f'  test:  node_dim={test_graph.x.shape[1]}')
print('RWPE done.')

def describe_graph(g, name):
    labels = g.y[g.eval_mask]
    n_pos  = (labels == 1).sum().item()
    n_eval = g.eval_mask.sum().item()
    print(f'{name}: nodes={g.num_nodes:,}  edges={g.edge_index.shape[1]:,}  '
          f'eval={n_eval:,}  laund={n_pos:,} ({100*n_pos/n_eval:.4f}%)  '
          f'node_dim={g.x.shape[1]}  edge_dim={g.edge_attr.shape[1]}')

describe_graph(train_graph, 'train_graph')
describe_graph(val_graph,   'val_graph')
describe_graph(test_graph,  'test_graph')

for g, name in [(train_graph,'train'),(val_graph,'val'),(test_graph,'test')]:
    assert (g.y[g.eval_mask] == -1).sum() == 0
print('Label sanity check passed.')


Computing RWPE per snapshot...
  train: node_dim=13
  val:   node_dim=13
  test:  node_dim=13
RWPE done.
train_graph: nodes=712,684  edges=4,154,429  eval=4,154,429  laund=1,813 (0.0436%)  node_dim=13  edge_dim=16
val_graph: nodes=712,684  edges=5,539,239  eval=1,384,810  laund=827 (0.0597%)  node_dim=13  edge_dim=16
test_graph: nodes=712,684  edges=6,924,049  eval=1,384,810  laund=925 (0.0668%)  node_dim=13  edge_dim=16
Label sanity check passed.


## 4. Hyperparameters

All training and model hyperparameters are defined here in one place for easy tuning.

### Alignment with the Paper

The following settings match the paper exactly (Appendix E, Table 11, Table 12):
- `HIDDEN_DIM = 64` -- hidden embedding size
- `NUM_LAYERS = 2` -- number of GNN message passing layers
- `NUM_NEIGHBORS = [100, 100]` -- 100 one-hop and 100 two-hop neighbours sampled per seed edge

### Class Imbalance Handling

With a 2,290:1 ratio of legitimate to laundering transactions, the model would simply predict everything as legitimate without correction. We use `pos_weight = 50` in the Binary Cross-Entropy loss, which means every laundering transaction contributes 50x more to the loss than a legitimate one. This forces the model to pay attention to the rare class.

The paper used `pos_weight` in the range (6, 8). We use 50 because we train on the real distribution (no oversampling), which requires a stronger correction.

In [5]:
NODE_DIM = train_graph.x.shape[1]          # 13 (5 original + 8 RWPE)
EDGE_DIM = train_graph.edge_attr.shape[1]  # 16 (no GFP)

HIDDEN_DIM    = 64
NUM_LAYERS    = 2
DROPOUT       = 0.3
EPOCHS        = 20
LR            = 1e-3
WEIGHT_DECAY  = 1e-5
NUM_NEIGHBORS = [100, 100]
BATCH_SIZE    = 2048

n_neg = (train_graph.y[train_graph.eval_mask] == 0).sum().item()
n_pos = (train_graph.y[train_graph.eval_mask] == 1).sum().item()
print(f'Train imbalance: {n_neg/n_pos:.0f}:1')

POS_WEIGHT = torch.tensor([8.0], device=device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
print(f'NODE_DIM={NODE_DIM}  EDGE_DIM={EDGE_DIM}  HIDDEN_DIM={HIDDEN_DIM}  EPOCHS={EPOCHS}')


Train imbalance: 2290:1
NODE_DIM=13  EDGE_DIM=16  HIDDEN_DIM=64  EPOCHS=20


## 5. Model Architecture

### The Core Problem: Classifying Edges, Not Nodes

Standard GNNs produce **node embeddings** through message passing. But our task is to classify **edges** (transactions). This requires a two-step pattern:

1. **Encode** -- run message passing over the neighbourhood graph to build informative node embeddings
2. **Decode** -- for each transaction we want to classify, look up the sender and receiver embeddings and combine them with the transaction's own features to produce a classification

This encode-decode separation is also required by how PyG's `LinkNeighborLoader` works. It keeps two sets of edges in each batch:
- `batch.edge_index` -- context edges used only for message passing (no labels)
- `batch.edge_label_index` -- seed edges that we actually want to classify (have labels)



### Why LayerNorm Instead of BatchNorm

BatchNorm normalises across the entire batch. With ~200,000 context edges per batch but only ~1 laundering edge on average, the batch mean and variance are completely dominated by legitimate transactions. LayerNorm normalises each node independently across its 64 features, so laundering nodes keep their distinctive activation patterns regardless of the class distribution in the batch.

### GINe Architecture Overview

The full forward pass for one transaction A to B looks like this:

```
INPUT
  x [N, 5]         -- raw node features for all accounts
  edge_attr [E, 16] -- raw edge features for context transactions
  edge_label_attr [n_seeds, 16] -- raw features of the seed transactions

ENCODE (runs on the context graph to build node embeddings)
  node_proj:  x [N,5]          --> h0 [N, 64]
  edge_proj:  edge_attr [E,16] --> e  [E, 64]

  Layer 1 (GINEConv):
    For each account v:
      msg = SUM over neighbours u: ReLU(h0[u] + e[u->v])
      h1[v] = MLP(h0[v] + msg)  -- 64->128->64
    Then: LayerNorm -> ReLU -> Dropout

  Layer 2 (GINEConv): same as layer 1, using h1
    Produces h2 [N, 64]

DECODE (classifies the seed transactions)
  For transaction A -> B:
    e_seed = edge_proj(edge_label_attr) --> [n_seeds, 64]
    edge_emb = concat(h2[A], h2[B], e_seed) --> [n_seeds, 192]
    logit = MLP(edge_emb)  -- 192->64->1
    P(laundering) = sigmoid(logit)
```

The three components in the decoder each carry different information:
- `h2[A]` -- what kind of sender account A is (its full 2-hop neighbourhood context)
- `h2[B]` -- what kind of receiver account B is
- `e_seed` -- the specific features of this particular transaction (amount, currency mismatch, payment format, time of day)

Without `e_seed`, the model would assign the same score to every transaction between the same pair of accounts, regardless of the transaction amount or format.

In [6]:
def build_mlp(in_dim, hidden_dim, out_dim, num_layers=2, dropout=0.3):
    """
    Build a multi-layer perceptron (MLP) with LayerNorm and Dropout.

    This function is used in two places:
    1. As the update function inside each GINEConv layer
    2. As the final edge classifier in the decoder

    The last linear layer has no activation or normalisation, because
    the output is either fed into the next layer (which has its own activation)
    or is a raw logit that will be passed to sigmoid/BCE loss.

    Parameters
    ----------
    in_dim     : input feature dimension
    hidden_dim : width of intermediate layers
    out_dim    : output feature dimension
    num_layers : total number of linear layers
    dropout    : fraction of features randomly zeroed during training

    Returns
    -------
    nn.Sequential : the constructed MLP
    """
    layers = []
    dims   = [in_dim] + [hidden_dim] * (num_layers - 1) + [out_dim]

    for i in range(len(dims) - 1):
        layers.append(nn.Linear(dims[i], dims[i+1]))
        # Add activation, normalisation, and dropout after every layer except the last
        if i < len(dims) - 2:
            layers.append(nn.ReLU())
            layers.append(nn.LayerNorm(dims[i+1]))
            layers.append(nn.Dropout(dropout))

    return nn.Sequential(*layers)

In [7]:
'''
GINEConv (handles graph structure):
┌──────────────────────────────────────────────────┐
│  For each node v:                                │
│    agg = SUM[ ReLU(h[u] + e_{uv}) ]             │
│    input = h[v] + agg                            │
│                                                  │
│    YOUR mlp(input):  ◄── build_mlp lives here   │
│    ┌────────────────────────────────────────┐    │
│    │ Linear(64→128) → ReLU → LN → Dropout  │    │
│    │ → Linear(128→64)                       │    │
│    └────────────────────────────────────────┘    │
│                                                  │
│    h_new[v] = mlp output                        │
└──────────────────────────────────────────────────┘
'''

'\nGINEConv (handles graph structure):\n┌──────────────────────────────────────────────────┐\n│  For each node v:                                │\n│    agg = SUM[ ReLU(h[u] + e_{uv}) ]             │\n│    input = h[v] + agg                            │\n│                                                  │\n│    YOUR mlp(input):  ◄── build_mlp lives here   │\n│    ┌────────────────────────────────────────┐    │\n│    │ Linear(64→128) → ReLU → LN → Dropout  │    │\n│    │ → Linear(128→64)                       │    │\n│    └────────────────────────────────────────┘    │\n│                                                  │\n│    h_new[v] = mlp output                        │\n└──────────────────────────────────────────────────┘\n'

In [8]:
'''

h⁰[v]
  ↓
GINEConv:
  agg    = SUM[ ReLU(h⁰[u] + e_{uv}) ]
  h_new  = mlp( h⁰[v] + agg )         ← Linear→ReLU→LN→Drop→Linear
  ↓
LayerNorm(h_new)                        ← normalise per node across 64 features
  ↓
ReLU                                    ← clip negatives, add non-linearity
  ↓
Dropout(0.3)                            ← randomly zero 30% of features
  ↓
h¹[v]  — ready for next layer or decode

'''

'\n\nh⁰[v]\n  ↓\nGINEConv:\n  agg    = SUM[ ReLU(h⁰[u] + e_{uv}) ]\n  h_new  = mlp( h⁰[v] + agg )         ← Linear→ReLU→LN→Drop→Linear\n  ↓\nLayerNorm(h_new)                        ← normalise per node across 64 features\n  ↓\nReLU                                    ← clip negatives, add non-linearity\n  ↓\nDropout(0.3)                            ← randomly zero 30% of features\n  ↓\nh¹[v]  — ready for next layer or decode\n\n'

In [9]:
'''

h⁰ → [GINEConv + mlp(ReLU#1)] → LayerNorm → ReLU#2 → Dropout → h¹
h¹ → [GINEConv + mlp(ReLU#1)] → LayerNorm → ReLU#2 → Dropout → h²
                                                                    ↓
                                                                 decode()
'''

'\n\nh⁰ → [GINEConv + mlp(ReLU#1)] → LayerNorm → ReLU#2 → Dropout → h¹\nh¹ → [GINEConv + mlp(ReLU#1)] → LayerNorm → ReLU#2 → Dropout → h²\n                                                                    ↓\n                                                                 decode()\n'

In [10]:
class GINe(nn.Module):
    """
    Graph Isomorphism Network with Edge Features and Edge Readout.

    This implements the GIN baseline from Altman et al. (2023), which uses:
    - GINEConv layers for message passing (edge features modulate messages)
    - An edge readout decoder that combines sender embedding, receiver
      embedding, and the transaction's own features for classification

    The model follows an encode -> decode pattern:
    - encode(): runs message passing to produce node embeddings
    - decode(): classifies seed edges using those embeddings

    Parameters
    ----------
    node_dim   : number of raw node features (5)
    edge_dim   : number of raw edge features (16)
    hidden_dim : size of all hidden embeddings (64)
    num_layers : number of GNN message passing layers (2)
    dropout    : dropout rate applied after each GNN layer
    """

    def __init__(self, node_dim, edge_dim, hidden_dim, num_layers, dropout=0.3):
        super().__init__()
        self.dropout = dropout

        # Project raw node and edge features into the shared 64-dimensional space.
        # Both use the same dimensionality so they can be added together during
        # message passing: ReLU(h[u] + e_{uv}).
        # Note: edge_proj is reused in decode() to project seed edge features.
        self.node_proj = nn.Linear(node_dim, hidden_dim)
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)

        # Stack of GINEConv layers -- one per message passing round.
        # Each conv layer receives the MLP that will transform the aggregated messages.
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            # The MLP inside GINEConv: transforms (node_embedding + aggregated_messages)
            mlp = build_mlp(hidden_dim, hidden_dim * 2, hidden_dim, dropout=dropout)
            self.convs.append(GINEConv(mlp, edge_dim=hidden_dim))
            # LayerNorm applied after each GNN layer (NOT BatchNorm -- see Section 5 explanation)
            self.norms.append(nn.LayerNorm(hidden_dim))

        # Edge classifier: takes a 192-dim vector (64 + 64 + 64) and produces a single logit.
        # The three components are: sender embedding, receiver embedding, transaction features.
        self.edge_classifier = build_mlp(
            in_dim     = hidden_dim * 3,  # concat(h[src], h[dst], e_seed) = 3 x 64
            hidden_dim = hidden_dim,
            out_dim    = 1,
            dropout    = dropout,
        )

    def encode(self, x, edge_index, edge_attr):
        """
        Run message passing over the context subgraph to produce node embeddings.

        This function processes the CONTEXT edges (batch.edge_index), which are
        the neighbourhood edges sampled around the seed edges. They provide the
        structural context needed to build informative account embeddings.

        After 2 layers, each account embedding h2[v] encodes:
        - The account's own features
        - Features of accounts that transacted directly with it (1-hop)
        - Features of accounts two steps away (2-hop)

        Parameters
        ----------
        x          : node feature matrix [N, 5]
        edge_index : context edge connectivity [2, E]
        edge_attr  : context edge features [E, 16]

        Returns
        -------
        h : node embedding matrix [N, 64]
        """
        # Project raw features into the hidden space
        h = F.relu(self.node_proj(x))          # [N, 5]  -> [N, 64]
        e = F.relu(self.edge_proj(edge_attr))  # [E, 16] -> [E, 64]

        # Two rounds of message passing
        for conv, norm in zip(self.convs, self.norms):
            # GINEConv aggregation: for each node v, compute
            # h_new[v] = MLP(h[v] + SUM over u in N(v): ReLU(h[u] + e_{uv}))
            h = conv(h, edge_index, e)
            h = norm(h)  # normalise per node across its 64 features
            h = F.relu(h)  # add non-linearity between layers
            h = F.dropout(h, p=self.dropout, training=self.training)

        return h  # final node embeddings h2 [N, 64]

    def decode(self, h, edge_label_index, edge_label_attr):
        """
        Classify seed transactions using node embeddings and transaction features.

        This function processes the SEED edges (batch.edge_label_index), which are
        the labelled transactions we want to classify. These are completely separate
        from the context edges used in encode().

        The classification input combines three sources of information:
        - h[src]: what kind of account the sender is (from message passing)
        - h[dst]: what kind of account the receiver is (from message passing)
        - e_seed: the specific features of this transaction (amount, format, time)

        Parameters
        ----------
        h                : node embeddings from encode() [N, 64]
        edge_label_index : seed edge endpoints [2, n_seeds]
        edge_label_attr  : raw features of seed edges [n_seeds, 16]

        Returns
        -------
        logits : classification scores [n_seeds] (before sigmoid)
        """
        src, dst = edge_label_index  # source and destination account indices

        # Project seed edge features using the same projection as in encode().
        # This ensures edge features are always embedded in the same space.
        e_seed = F.relu(self.edge_proj(edge_label_attr))  # [n_seeds, 64]

        # Build the full edge representation by concatenating all three components
        edge_emb = torch.cat([h[src], h[dst], e_seed], dim=-1)  # [n_seeds, 192]

        # Final classification: 192 -> 64 -> 1
        return self.edge_classifier(edge_emb).squeeze(-1)

    def forward(self, x, edge_index, edge_attr, edge_label_index, edge_label_attr):
        """
        Full forward pass: encode the graph, then decode the seed edges.

        Parameters
        ----------
        x                : node features [N, 5]
        edge_index       : context edge connectivity [2, E] (for message passing)
        edge_attr        : context edge features [E, 16] (for message passing)
        edge_label_index : seed edge endpoints [2, n_seeds] (edges to classify)
        edge_label_attr  : seed edge features [n_seeds, 16] (edges to classify)

        Returns
        -------
        logits : classification scores [n_seeds]
        """
        h = self.encode(x, edge_index, edge_attr)
        return self.decode(h, edge_label_index, edge_label_attr)

## 6. Data Loader

### Why We Cannot Load the Full Graph at Once

The training graph has 4.15 million edges. Loading all of them for a single forward pass would require roughly 3 GB of GPU memory just for the activations and gradients. Instead, we use PyG's `LinkNeighborLoader` to process the graph in mini-batches.

For each mini-batch, the loader:
1. Picks a batch of seed edges (the transactions we want to classify)
2. For each seed edge's endpoints, samples a local neighbourhood (100 one-hop + 100 two-hop neighbours)
3. Builds a small subgraph from those sampled neighbours
4. Returns the subgraph with two edge sets:
   - `batch.edge_index`: the neighbourhood context edges (used for message passing)
   - `batch.edge_label_index`: the seed edges (used for classification and loss)

### How Seed Edge Features Are Tracked

`make_loader` returns both the loader and `seed_edge_attr` (the raw features for all seed edges). Inside each batch, `batch.input_id` tells us which seed edges from the full pool ended up in this batch, so we can fetch the correct features with `seed_edge_attr[batch.input_id.cpu()]`.

### Class Imbalance in Batches

With 0.0436% laundering rate and batch size 16,384, each batch sees on average only **7 laundering transactions** out of 16,384 total. The `pos_weight=50` in the loss function compensates for this by amplifying the gradient from those 7 edges.

In [11]:
def make_loader(graph, shuffle=True, verbose=False):
    """
    Build a LinkNeighborLoader for mini-batch edge classification.

    The loader uses the real class distribution (no oversampling). Class
    imbalance is handled instead through pos_weight in the loss function.

    Returns a tuple of (loader, seed_edge_attr) because the loader itself
    does not carry seed edge features -- they must be looked up separately
    using batch.input_id in the training loop.

    Parameters
    ----------
    graph   : PyG Data object (train_graph, val_graph, or test_graph)
    shuffle : True during training for randomness; False during evaluation
    verbose : if True, print the number of positive and negative seed edges

    Returns
    -------
    loader         : LinkNeighborLoader that yields mini-batches
    seed_edge_attr : raw edge features for all seed edges [n_seeds, 16]
    """
    # Extract only the labelled edges from this graph snapshot.
    # eval_mask marks which edges belong to this split's evaluation set.
    seed_mask       = graph.eval_mask
    seed_edge_index = graph.edge_index[:, seed_mask]  # [2, n_seeds]
    seed_labels     = graph.y[seed_mask].float()       # [n_seeds] -- 0.0 or 1.0
    seed_edge_attr  = graph.edge_attr[seed_mask]       # [n_seeds, 16] -- returned separately

    if verbose:
        n_pos = (seed_labels == 1).sum().item()
        n_neg = (seed_labels == 0).sum().item()
        print(f'  Seed edges : {n_pos + n_neg:,} total')
        print(f'  Laundering : {n_pos:,} ({100*n_pos/(n_pos+n_neg):.4f}%)')

    loader = LinkNeighborLoader(
        data             = graph,           # full graph (for neighbourhood sampling)
        num_neighbors    = NUM_NEIGHBORS,   # [100, 100] matches paper
        edge_label_index = seed_edge_index, # which edges to classify
        edge_label       = seed_labels,     # their labels (0 or 1)
        batch_size       = BATCH_SIZE,      # seed edges per mini-batch
        shuffle          = shuffle,
        num_workers      = 0,               # must be 0 in Colab (multiprocessing issues)
        pin_memory       = False,
    )

    return loader, seed_edge_attr

## 7. Training and Evaluation

### Training Loop (`train_epoch`)

Each epoch iterates over all mini-batches produced by the loader. For each batch:
1. The context subgraph is encoded to produce node embeddings
2. The seed edge features are fetched using `batch.input_id` as an index into `seed_edge_attr`
3. The decoder classifies each seed edge using sender embedding + receiver embedding + edge features
4. BCE loss is computed with `pos_weight=50` to upweight laundering edges
5. Gradients are clipped to prevent instability, then weights are updated

### Evaluation Loop (`evaluate`)

Evaluation collects predicted probabilities and true labels across all batches, then computes metrics at a fixed threshold. The threshold default is 0.5, but this is not optimal for imbalanced data -- threshold tuning is done separately in Section 9.

### Checkpointing Strategy

We checkpoint based on **validation AUC**. AUC is threshold-independent and more stable for severely imbalanced datasets.

### Learning Rate Schedule

Cosine annealing smoothly reduces the learning rate from the initial value down to `eta_min=1e-5` over the training run. This prevents overshooting at the end of training and typically improves final performance.

In [12]:
# Loss function with class imbalance correction.
# BCEWithLogitsLoss combines sigmoid + binary cross-entropy in one numerically stable operation.
# pos_weight=50 means laundering edges contribute 50x more loss than legitimate ones.
criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)


def train_epoch(model, graph, optimizer):
    """
    Run one full training epoch over all seed edges in the graph.

    Iterates over mini-batches from the loader. For each batch, performs
    a forward pass, computes loss, and updates model weights.

    Parameters
    ----------
    model     : GINe model instance
    graph     : training graph (train_graph)
    optimizer : Adam optimiser

    Returns
    -------
    float : average BCE loss across all batches in this epoch
    """
    model.train()
    loader, seed_edge_attr = make_loader(graph, shuffle=True, verbose=True)
    total_loss = 0.0
    n_batches  = 0
    pbar       = tqdm(loader, desc='  Training', leave=False)

    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()

        # Fetch the raw edge features for the seed edges in this batch.
        # batch.input_id contains the positions of this batch's seed edges
        # in the full seed pool returned by make_loader.
        # We index on CPU then move to GPU to avoid device mismatch errors.
        seed_attr = seed_edge_attr[batch.input_id.cpu()].to(device)

        # Forward pass: encode context graph, decode seed edges
        logits = model(
            batch.x,
            batch.edge_index,        # context edges for message passing
            batch.edge_attr,         # context edge features
            batch.edge_label_index,  # seed edges to classify
            seed_attr,               # seed edge features
        )

        # Compute loss against true labels (0=legitimate, 1=laundering)
        loss = criterion(logits, batch.edge_label)
        loss.backward()

        # Clip gradients to prevent exploding gradients on the sparse laundering signal
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(model, graph, threshold=0.5):
    """
    Evaluate the model on a graph snapshot and return classification metrics.

    Uses the real class distribution (no oversampling) so that metrics
    reflect true performance on the original data. The threshold parameter
    controls the boundary between predicted laundering and legitimate.

    Note: threshold=0.5 is used here for monitoring during training.
    The optimal threshold is found separately using find_best_threshold()
    after training completes.

    Parameters
    ----------
    model     : trained GINe model
    graph     : graph to evaluate on (val_graph or test_graph)
    threshold : decision boundary for converting probabilities to predictions

    Returns
    -------
    dict with keys: f1, precision, recall, auc
    """
    model.eval()
    loader, seed_edge_attr = make_loader(graph, shuffle=False)
    all_probs  = []
    all_labels = []
    pbar       = tqdm(loader, desc='  Validation', leave=False)

    for batch in pbar:
        batch     = batch.to(device)
        seed_attr = seed_edge_attr[batch.input_id.cpu()].to(device)

        logits = model(
            batch.x,
            batch.edge_index,
            batch.edge_attr,
            batch.edge_label_index,
            seed_attr,
        )
        # Convert logits to probabilities and collect across batches
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(batch.edge_label.long().cpu().numpy())

    # Concatenate all batches into single arrays
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    # Apply threshold to get binary predictions
    preds = (all_probs >= threshold).astype(int)

    # Compute minority-class metrics (pos_label=1 means we evaluate on laundering class only)
    f1  = f1_score(all_labels, preds, pos_label=1, zero_division=0)
    pre = precision_score(all_labels, preds, pos_label=1, zero_division=0)
    rec = recall_score(all_labels, preds, pos_label=1, zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        # This can happen if a batch has no positives -- safe fallback
        auc = float('nan')

    return {'f1': f1, 'precision': pre, 'recall': rec, 'auc': auc}


def run_training(model, model_name, checkpoint_path, epochs=EPOCHS):
    """
    Full training loop with validation monitoring and model checkpointing.

    Trains the model for the specified number of epochs, evaluating on the
    validation set after each epoch. Saves the model state whenever validation
    AUC improves. At the end, loads the best checkpoint and evaluates on the
    test set.

    We checkpoint on AUC rather than F1 because AUC is threshold-independent
    and more reliable when F1 at threshold=0.5 is often 0.0 early in training
    (the model outputs low probabilities before it has learned to discriminate).

    Parameters
    ----------
    model           : GINe model instance
    model_name      : name string used in printed output
    checkpoint_path : file path to save the best model weights
    epochs          : number of training epochs

    Returns
    -------
    model        : model loaded with best checkpoint weights
    history      : list of dicts with per-epoch metrics
    test_metrics : dict with final test set results
    """
    optimizer = Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # Cosine annealing smoothly reduces learning rate to eta_min over all epochs.
    # This avoids overshooting at the end of training.
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    best_val_auc = 0.0
    best_state   = None
    history      = []

    print(f'\n{"="*60}')
    print(f'Training {model_name}')
    print(f'  Parameters : {sum(p.numel() for p in model.parameters()):,}')
    print(f'  pos_weight : 50.0 (compensates for no oversampling)')
    print(f'{"="*60}')

    for epoch in range(1, epochs + 1):
        print(f'\n--- Epoch {epoch}/{epochs} ---')
        t0 = time.time()

        # Training step
        train_loss = train_epoch(model, train_graph, optimizer)
        scheduler.step()  # update the learning rate for the next epoch

        # Validation step
        val_metrics = evaluate(model, val_graph)
        elapsed     = time.time() - t0

        history.append({'epoch': epoch, 'loss': train_loss, **val_metrics})

        # Save model if validation AUC improved
        improved = ''
        if val_metrics['auc'] > best_val_auc:
            best_val_auc = val_metrics['auc']
            # Deep copy the state so future epochs do not overwrite it
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
            torch.save(model.state_dict(), checkpoint_path)
            improved     = '  --> New Best Model!'

        print(
            f'Result: Train Loss: {train_loss:.4f} | '
            f'Val F1: {val_metrics["f1"]:.4f} | '
            f'Val Pre: {val_metrics["precision"]:.4f} | '
            f'Val Rec: {val_metrics["recall"]:.4f} | '
            f'Val AUC: {val_metrics["auc"]:.4f} | '
            f'Time: {elapsed:.1f}s'
            f'{improved}'
        )

    # Restore best checkpoint for final evaluation
    if best_state is not None:
        model.load_state_dict(best_state)

    # Final evaluation on the held-out test set
    test_metrics = evaluate(model, test_graph)
    print(f'\n{"="*60}')
    print(f'Final Test Results -- {model_name}')
    print(f'  F1        : {test_metrics["f1"]:.4f}')
    print(f'  Precision : {test_metrics["precision"]:.4f}')
    print(f'  Recall    : {test_metrics["recall"]:.4f}')
    print(f'  AUC-ROC   : {test_metrics["auc"]:.4f}')
    print(f'{"="*60}')

    return model, history, test_metrics

## 8. Training the Model

We initialise GINe and train it for 10 epochs. Each epoch takes roughly 3-4 minutes on a T4 GPU.

**What to expect during training:**

- **Val F1 = 0.0 in early epochs** -- this is normal and not a bug. With only ~7 laundering edges per batch and the model outputting very low probabilities initially, nothing crosses the default threshold of 0.5. AUC is the more informative metric during training because it measures how well the model ranks laundering transactions above legitimate ones, regardless of threshold.
- **AUC should climb consistently each epoch** -- if it stays flat or drops, something is wrong.
- **Loss decreasing does not necessarily mean the model is learning** -- with 99.96% legitimate edges, a model that predicts zero for everything has very low loss but is completely useless. AUC is the honest metric here.

The model is saved to Google Drive whenever validation AUC improves, so training can be resumed if the Colab session expires.

In [13]:
torch.manual_seed(SEED)
gine_rwpe_model = GINe(
    node_dim   = NODE_DIM,   # 13
    edge_dim   = EDGE_DIM,   # 16
    hidden_dim = HIDDEN_DIM,
    num_layers = NUM_LAYERS,
    dropout    = DROPOUT,
).to(device)

gine_rwpe_model, gine_rwpe_history, gine_rwpe_test = run_training(
    gine_rwpe_model,
    model_name      = 'GINe + RWPE',
    checkpoint_path = '/kaggle/working/Models/GINe_rwpe/gine_rwpe_epochs_20.pt',
    epochs          = EPOCHS,
)



Training GINe + RWPE
  Parameters : 56,769
  pos_weight : 50.0 (compensates for no oversampling)

--- Epoch 1/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0190 | Val F1: 0.0000 | Val Pre: 0.0000 | Val Rec: 0.0000 | Val AUC: 0.9359 | Time: 179.1s  --> New Best Model!

--- Epoch 2/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0164 | Val F1: 0.0000 | Val Pre: 0.0000 | Val Rec: 0.0000 | Val AUC: 0.9506 | Time: 178.5s  --> New Best Model!

--- Epoch 3/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0157 | Val F1: 0.0000 | Val Pre: 0.0000 | Val Rec: 0.0000 | Val AUC: 0.9459 | Time: 178.5s

--- Epoch 4/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0151 | Val F1: 0.0672 | Val Pre: 0.3263 | Val Rec: 0.0375 | Val AUC: 0.9593 | Time: 179.3s  --> New Best Model!

--- Epoch 5/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


...

Result: Train Loss: 0.0123 | Val F1: 0.1143 | Val Pre: 0.4074 | Val Rec: 0.0665 | Val AUC: 0.9653 | Time: 174.7s

--- Epoch 18/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0122 | Val F1: 0.1271 | Val Pre: 0.3841 | Val Rec: 0.0762 | Val AUC: 0.9654 | Time: 174.7s

--- Epoch 19/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0121 | Val F1: 0.1250 | Val Pre: 0.4094 | Val Rec: 0.0738 | Val AUC: 0.9655 | Time: 175.7s

--- Epoch 20/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0121 | Val F1: 0.1282 | Val Pre: 0.4038 | Val Rec: 0.0762 | Val AUC: 0.9661 | Time: 176.2s



Final Test Results -- GINe + RWPE
  F1        : 0.1765
  Precision : 0.4256
  Recall    : 0.1114
  AUC-ROC   : 0.9668


In [14]:
# Optional: load a previously saved checkpoint.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gine_rwpe_model = GINe(
    node_dim=NODE_DIM, edge_dim=EDGE_DIM,
    hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
).to(device)

checkpoint_path = '/kaggle/working/Models/GINe_rwpe/gine_rwpe_epochs_20.pt'
if os.path.exists(checkpoint_path):
    gine_rwpe_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print('Model loaded.')
else:
    print('Checkpoint not found.')


Model loaded.


## 9. Save Predictions for All Splits

In [15]:
@torch.no_grad()
def score_split(model, graph, split_name):
    model.eval()
    loader, seed_edge_attr = make_loader(graph, shuffle=False)
    all_probs, all_labels, all_input_ids = [], [], []
    for batch in tqdm(loader, desc=f'  Scoring {split_name}', leave=False):
        batch = batch.to(device)
        seed_attr = seed_edge_attr[batch.input_id.cpu()].to(device)
        logits = model(batch.x, batch.edge_index, batch.edge_attr,
                       batch.edge_label_index, seed_attr)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(batch.edge_label.long().cpu().numpy())
        all_input_ids.append(batch.input_id.cpu().numpy())

    scores    = np.concatenate(all_probs)
    labels    = np.concatenate(all_labels)
    input_ids = np.concatenate(all_input_ids)
    eval_ei   = graph.edge_index[:, graph.eval_mask]
    eval_et   = graph.edge_time[graph.eval_mask]

    df = pd.DataFrame({
        'split': split_name,
        'src_idx': eval_ei[0][input_ids].numpy(),
        'dst_idx': eval_ei[1][input_ids].numpy(),
        'timestamp': eval_et[input_ids].numpy(),
        'score': scores, 'label': labels,
    })
    print(f'  {split_name}: {len(df):,} edges | {(labels==1).sum():,} laund | '
          f'score [{scores.min():.4f}, {scores.max():.4f}]')
    return df

import datetime
date_str = datetime.date.today().strftime('%d_%m_%y')
os.makedirs('/kaggle/working/Models/GINe_rwpe/Predictions', exist_ok=True)

df_train = score_split(gine_rwpe_model, train_graph, 'train')
df_val   = score_split(gine_rwpe_model, val_graph,   'val')
df_test  = score_split(gine_rwpe_model, test_graph,  'test')

df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)
out_path = f'/kaggle/working/Models/GINe_rwpe/Predictions/gine_rwpe_predictions_all_{date_str}.csv'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df_all.to_csv(out_path, index=False)
print(f'Saved -> {out_path}  ({len(df_all):,} rows)')

  train: 4,154,429 edges | 1,813 laund | score [0.0000, 0.9408]


  val: 1,384,810 edges | 827 laund | score [0.0000, 0.9309]


  test: 1,384,810 edges | 925 laund | score [0.0000, 0.9339]
Saved -> /kaggle/working/Models/GINe_rwpe/Predictions/gine_rwpe_predictions_all_25_05_26.csv  (6,924,049 rows)
